In [ ]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\msl2401.60270\OneDrive - IKNL\Documenten\FAIVOR\K21019_prepared_full.csv",
    low_memory=False,
)
df.head()

In [ ]:
for col in ["gesl", "gedrag", "menopstatus", "oestrrec_stat", "progrec_stat",
            "her2_uitslag", "first_event_5y", "first_event_10y",
            "pt", "pn", "cn", "ct", "chirurgie1", "fu10y_rectopo1"]:
    print(f"{col:16} {df[col].dropna().unique()[:12]}")

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# CONFIG: numeric codes as they appear in the data
# ============================================================
FEMALE        = 2           # gesl: 2 = Vrouw
MALIGNANT     = 3           # gedrag: 3 = malignant (ICD-O)
IN_SITU       = 2           # gedrag: 2 = in situ (ICD-O)
MENOP_POST    = [3.0, 4.0]  # menopstatus: 3 = post-menopausal, 4 = post (surgical)
# ============================================================


def prepare_ID960_Chen(data: pd.DataFrame) -> pd.DataFrame:
    df = data.copy()

    df = df.loc[df["gedrag"] == MALIGNANT]
    df = df.loc[(df["cm"] != 1) | (df["Meta1"].isna())]
    df = df.loc[~(df["pt"].eq("IS") | df["pt"].isna() | df["pn"].isna())].copy()

    df["outcome"] = np.where(
        (df["vitstat"] == 1) & (df["vitfup"] < 1825), 1, 0
    )
    df.loc[(df["vitstat"] == 0) & (df["vitfup"] < 1825), "outcome"] = np.nan
    df = df.loc[df["outcome"].notna()].copy()

    df = df[["leeft", "oestrrec_stat", "progrec_stat", "diffgr", "pt", "pn", "outcome"]].rename(
        columns={"outcome": "OS"}
    ).copy()
    return df


def prepare_ID14_Dowsett(data: pd.DataFrame, orig_data: pd.DataFrame | None = None) -> pd.DataFrame:
    df = data.copy()
    ref = orig_data if orig_data is not None else data

    mask = (
        ((ref["oestrrec_stat"] == 1) | (ref["oestrrec_stat"].isna()))
        & (ref["menopstatus"].isin(MENOP_POST) | ref["menopstatus"].isna())
    )
    df = df.loc[mask]
    df = df.loc[df["first_event_5y"] == "tumvrij"]
    df = df.loc[~((df["fu10y_rectopo1"] == "C50") & (df["fu10y_rectopo2"].isna()))]
    df = df.loc[df["gesl"] == FEMALE]
    df = df.loc[df["first_event_10y"] != "Geen follow-up"].copy()

    df["Recurrence_risk"] = np.where(df["fu10y_rectopo1"].isna(), 0, 1)

    df.loc[df["lypos"] == 3, "lypos"] = 2
    df.loc[(df["lypos"] >= 4) & (df["lypos"] <= 9), "lypos"] = 3
    df.loc[df["lypos"] > 9, "lypos"] = 4
    df.loc[df["tumorgrootte"] > 30, "tumorgrootte"] = 3

    df = df.loc[(df["inc_horm_start1"] < 1825) & (df["horm1"].notna())]
    df = df.loc[(df["oestrrec_stat"] == 1) | (df["progrec_stat"] == 1)]
    df = df.loc[df["menopstatus"].isin(MENOP_POST) | df["menopstatus"].isna()].copy()

    df = df[["tumorgrootte", "diffgr", "leeft", "lypos", "Recurrence_risk"]].copy()
    df = df.loc[df["lypos"].notna()].copy()
    return df


def prepare_ID721_Li(data: pd.DataFrame) -> pd.DataFrame:
    df = data.copy()

    df = df.loc[df["gedrag"] == MALIGNANT]
    df = df.loc[df["gesl"] == FEMALE]
    df = df.loc[df["chirurgie1"].isin(["130C50", "131C50", "132C50"])].copy()

    df["DFS"] = np.select(
        [df["first_event_5y"] == "Geen follow-up", df["first_event_5y"] == "tumvrij"],
        [np.nan, 1],
        default=0,
    )

    df = df.loc[df["DFS"].notna() & df["lypos"].notna()].copy()
    df = df[["oestrrec_stat", "progrec_stat", "her2_uitslag", "diffgr", "lypos", "DFS"]].copy()
    return df


def prepare_ID52_Zhang(data: pd.DataFrame) -> pd.DataFrame:
    df = data.copy()

    df = df.loc[df["swk_jn"] == 1]

    alnd_codes = ["132C50", "142C50"]
    chirurgie_cols = ["chirurgie1", "chirurgie2", "chirurgie3", "chirurgie4", "chirurgie5"]
    df = df.loc[df[chirurgie_cols].isin(alnd_codes).any(axis=1)]

    df = df.loc[df["basisd"] == 7]
    df = df.loc[df["gesl"] == FEMALE]
    df = df.loc[~((df["cn"] == "X") | (df["cn"].isna()))]
    df = df.loc[~df["ct"].isin(["4", "4A", "4B", "4C", "4D", "X"])]

    df = df[
        ["leeft", "ct", "gedrag", "topog", "morf", "cn",
         "her2_uitslag", "oestrrec_stat", "progrec_stat",
         "swk_uitslag", "lyond", "lypos"]
    ].copy()

    df["subtype"] = np.select(
        [
            (df["her2_uitslag"] == 0) & (df["oestrrec_stat"] == 0) & (df["progrec_stat"] == 0),
            (df["her2_uitslag"] == 0) & ((df["oestrrec_stat"] == 1) | (df["progrec_stat"] == 1)),
            (df["her2_uitslag"] == 1),
        ],
        ["TN", "Luminal", "HER2+"],
        default=None,
    )

    df["morf"] = np.select(
        [
            df["gedrag"] == IN_SITU,
            df["morf"].isin([8520, 8524]),
            df["morf"].isin([8500, 8514, 8523]),
        ],
        ["in situ", "Lobulair", "Ductaal"],
        default="Other",
    )

    df = df.loc[(df["lyond"] > 0) & (df["lyond"].notna())].copy()
    df["Ax_Lymph_node_involvement"] = np.where(df["lypos"] > 0, 1, 0)
    return df


def prepare_all_models(data: pd.DataFrame, orig_data: pd.DataFrame | None = None) -> dict[str, pd.DataFrame]:
    return {
        "ID960_Chen": prepare_ID960_Chen(data),
        "ID14_Dowsett": prepare_ID14_Dowsett(data, orig_data=orig_data),
        "ID721_Li": prepare_ID721_Li(data),
        "ID52_Zhang": prepare_ID52_Zhang(data),
    }

In [ ]:
from pathlib import Path

outdir = Path("output")
outdir.mkdir(exist_ok=True)

COLUMN_MAP = {                       # your spec: new name -> source column
    "Age": "leeft", "ER": "oestrrec_stat", "HER2": "her2_uitslag",
    "N_positive_node": "lypos", "Node_clin_grade": "cn", "Node_hist_grade": "pn",
    "PR": "progrec_stat", "Pathological_type": "morf", "Topography": "topog",
    "Tumor_clin_grade": "ct", "Tumor_grade": "diffgr", "Tumor_hist_grade": "pt",
    "Tumor_size": "tumorgrootte", "Invasive_disease": "gedrag",
}
rename_map = {src: new for new, src in COLUMN_MAP.items()}   # invert for df.rename

datasets = prepare_all_models(df)

for name, out in datasets.items():
    out = out.rename(columns=rename_map)
    path = outdir / f"{name}.csv"
    out.to_csv(path, index=False)
    print(f"{name}: {out.shape[0]} rows × {out.shape[1]} cols → {path}")